In [ ]:
import re
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from wordcloud import WordCloud

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

stop_words = set(stopwords.words("english"))

In [ ]:
paragraph = (
    "Artificial intelligence is transforming the way people learn and work. "
    "I enjoy exploring machine learning because it helps computers discover useful patterns. "
    "Python makes experimentation accessible through simple and powerful libraries. "
    "Responsible technology should protect privacy and support human decisions. "
    "I am especially interested in building practical applications that solve everyday problems. "
    "Learning new tools regularly helps me become a better developer."
)

lower_text = paragraph.lower()
clean_text = re.sub(r"[^a-z0-9\s]", " ", lower_text)
sentences = sent_tokenize(paragraph)
words = word_tokenize(clean_text)
split_words = clean_text.split()
nltk_words = word_tokenize(clean_text)
filtered_words = [w for w in nltk_words if w not in stop_words and w.isalpha()]
print(paragraph)
print(clean_text)
print(sentences)
print(split_words)
print(nltk_words)
print(filtered_words)
display(pd.DataFrame(Counter(filtered_words).most_common(), columns=["word", "frequency"]))

In [ ]:
alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", clean_text)
filtered_alpha = [w for w in alphabetic_words if w not in stop_words]
porter = PorterStemmer()
wnl = WordNetLemmatizer()
stems = [porter.stem(w) for w in filtered_alpha]
lemmas = [wnl.lemmatize(w) for w in filtered_alpha]
display(pd.DataFrame({"word": filtered_alpha, "Porter stem": stems, "WordNet lemma": lemmas}))

In [ ]:
texts = [
    "Artificial intelligence helps doctors analyze medical images and support diagnosis.",
    "Blockchain stores transactions securely across a distributed digital network.",
    "Machine learning systems identify patterns from data and improve predictions."
]
count_vec = CountVectorizer()
bow = count_vec.fit_transform(texts)
display(pd.DataFrame(bow.toarray(), columns=count_vec.get_feature_names_out(), index=["Text 1", "Text 2", "Text 3"]))
tfidf_vec = TfidfVectorizer(stop_words="english")
tfidf = tfidf_vec.fit_transform(texts)
terms = tfidf_vec.get_feature_names_out()
display(pd.DataFrame(tfidf.toarray(), columns=terms, index=["Text 1", "Text 2", "Text 3"]).round(3))
for i, row in enumerate(tfidf.toarray(), 1):
    print(f"Text {i}:", sorted(zip(terms, row), key=lambda x: x[1], reverse=True)[:3])

In [ ]:
tech_ai = """Artificial intelligence learns patterns from data.
It supports prediction, automation, and decision making.
AI systems can understand language and recognize images.
Machine learning is one approach used to build AI tools."""

tech_blockchain = """Blockchain records transactions in linked blocks.
It supports secure storage and decentralized verification.
Distributed networks maintain a shared transaction history.
Cryptographic methods help protect blockchain records."""

def preprocess(text):
    tokens = re.findall(r"\b[a-z]+\b", text.lower())
    return [t for t in tokens if t not in stop_words]

tokens_ai = preprocess(tech_ai)
tokens_blockchain = preprocess(tech_blockchain)
set_ai, set_blockchain = set(tokens_ai), set(tokens_blockchain)
jaccard = len(set_ai & set_blockchain) / len(set_ai | set_blockchain)
tf = TfidfVectorizer(stop_words="english")
matrix = tf.fit_transform([tech_ai, tech_blockchain])
cosine = cosine_similarity(matrix[0:1], matrix[1:2])[0, 0]
print(jaccard)
print(cosine)

In [ ]:
reviews = [
    "The product is excellent, easy to use, and works perfectly.",
    "I love the fast service and the helpful customer support.",
    "The quality is terrible and the device stopped working.",
    "Very disappointing experience; the app is slow and confusing.",
    "The product arrived today. It has a standard design."
]
sentiment_rows = []
positive_reviews = []
for review in reviews:
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    label = "Positive" if polarity > 0.1 else "Negative" if polarity < -0.1 else "Neutral"
    sentiment_rows.append({"review": review, "polarity": polarity, "subjectivity": subjectivity, "sentiment": label})
    if label == "Positive":
        positive_reviews.append(review)
display(pd.DataFrame(sentiment_rows))
positive_text = " ".join(positive_reviews)
if positive_text.strip():
    wc = WordCloud(width=900, height=450, background_color="white").generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

training_text = (
    "Technology changes the world through curiosity and creativity. "
    "People learn by testing ideas and building useful tools. "
    "Good software solves real problems and makes difficult tasks easier. "
    "Data helps systems discover patterns and improve their predictions. "
    "Responsible innovation considers safety, fairness, and human needs. "
    "Developers practice regularly to improve their skills and create better applications. "
    "Learning is a continuous process that connects knowledge with experience."
)
tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
sequences = []
for line in training_text.split("."):
    seq = tokenizer.texts_to_sequences([line.strip()])[0]
    for i in range(2, len(seq) + 1):
        sequences.append(seq[:i])
max_len = max(len(s) for s in sequences)
padded = pad_sequences(sequences, maxlen=max_len, padding="pre")
X, y = padded[:, :-1], padded[:, -1]
model = Sequential([
    Embedding(total_words, 32, input_length=max_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X, y, epochs=100, verbose=0)

def generate_text(seed, next_words=12):
    result = seed
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([result])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding="pre")
        prediction = model.predict(seq, verbose=0).argmax(axis=1)[0]
        next_word = tokenizer.index_word.get(prediction, "")
        if not next_word:
            break
        result += " " + next_word
    return result

for seed in ["technology", "learning", "data"]:
    print(generate_text(seed, 12))